# 08 — + Centroid Projection (All Hidden Layers)

Builds on **07 +Centroid(Last)**: adds CentroidProjection after hidden_1 as well.
Double projection provides stronger geometric regularization throughout the network.

**Cumulative stack:** L2 + BN + LN + Dropout + L1 + Centroid(all)

**What to look for:**
- Both proj_1 and proj_2 layers should show tight clusters
- Compare to 07 — does projecting both layers help more than just the last?
- Centroids are computed globally from full dataset at the start of each epoch

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from scripts.viz_export import ExperimentTracker

torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__}")
device = torch.device("cpu")

## 1. Load MNIST

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1)),
])

data_root = os.path.join(PROJECT_ROOT, "data")
full_train_dataset = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    full_train_dataset, [48000, 12000],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, generator=torch.Generator().manual_seed(0))
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 48000, Val: 12000, Test: 10000


## 2. Define Model

In [3]:
PROJ_ALPHA = 0.3
PROJ_BETA = 0.1

class CentroidProjection(nn.Module):
    """Directly manipulates activations: repel from other centroids, condense toward own.

    Centroids are computed globally after each epoch by calling update_centroids(),
    then used as fixed targets during the next epoch's forward passes.
    Straight-through for gradients (applied in forward, invisible to backward).
    """
    def __init__(self, dim, num_classes=10, alpha=0.1, beta=0.05):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes
        self.register_buffer('centroids', torch.zeros(num_classes, dim))
        self.register_buffer('ready', torch.tensor(False))

    def update_centroids(self, centroids):
        """Set centroids from a pre-computed [num_classes, dim] tensor."""
        self.centroids.copy_(centroids)
        self.ready.fill_(True)

    def forward(self, x, labels=None):
        if labels is None or not self.ready:
            return x

        with torch.no_grad():
            repulsion = torch.zeros_like(self.centroids)
            for c in range(self.num_classes):
                others = [i for i in range(self.num_classes) if i != c]
                diffs = self.centroids[c] - self.centroids[others]
                dists = diffs.norm(dim=1, keepdim=True).clamp(min=1e-6)
                repulsion[c] = (diffs / dists ** 2).sum(dim=0)

            rep_norms = repulsion.norm(dim=1, keepdim=True).clamp(min=1e-6)
            repulsion = repulsion / rep_norms * self.beta
            nudge = repulsion[labels]

            diff = self.centroids[labels] - x
            dist = diff.norm(dim=1, keepdim=True).clamp(min=1e-6)
            mean_dist = dist.mean()
            scale = dist / (mean_dist + 1e-8)
            nudge = nudge + self.alpha * scale * diff

        return x + nudge

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)
        self.proj1 = CentroidProjection(128, num_classes=10, alpha=PROJ_ALPHA, beta=PROJ_BETA)
        self.fc2 = nn.Linear(128, 64)
        self.ln2 = nn.LayerNorm(64)
        self.drop2 = nn.Dropout(0.3)
        self.proj2 = CentroidProjection(64, num_classes=10, alpha=PROJ_ALPHA, beta=PROJ_BETA)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x, labels=None):
        x = self.drop1(torch.relu(self.bn1(self.fc1(x))))
        x = self.proj1(x, labels)
        x = self.drop2(torch.relu(self.ln2(self.fc2(x))))
        x = self.proj2(x, labels)
        x = self.fc3(x)
        return x

    def compute_centroids(self, loader, device):
        """Forward pass over full dataset to compute per-class mean activations at both hidden layers."""
        self.eval()
        sums1 = torch.zeros(10, 128, device=device)
        sums2 = torch.zeros(10, 64, device=device)
        counts = torch.zeros(10, device=device)
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x = batch_x.to(device)
                h1 = self.drop1(torch.relu(self.bn1(self.fc1(batch_x))))
                h2 = self.drop2(torch.relu(self.ln2(self.fc2(h1))))
                for c in range(10):
                    mask = (batch_y == c)
                    if mask.any():
                        sums1[c] += h1[mask].sum(dim=0)
                        sums2[c] += h2[mask].sum(dim=0)
                        counts[c] += mask.sum()
        counts = counts.unsqueeze(1).clamp(min=1)
        return sums1 / counts, sums2 / counts

model = MLP().to(device)
print(f"CentroidProjection: alpha={PROJ_ALPHA}, beta={PROJ_BETA}")
print(model)

CentroidProjection: alpha=0.3, beta=0.1
MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (proj1): CentroidProjection()
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (proj2): CentroidProjection()
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)


## 3. Pick Input Samples & Viz Samples

In [4]:
# 5 input samples: one per distinct digit (first 5 unique)
seen_labels = set()
five_images, five_labels = [], []
for img, label in full_train_dataset:
    if label not in seen_labels:
        five_images.append(img)
        five_labels.append(int(label))
        seen_labels.add(label)
    if len(seen_labels) == 5:
        break
five_images = torch.stack(five_images)
print(f"Input samples: labels={five_labels}, shape={five_images.shape}")

# 500 viz samples: 50 per class
viz_images, viz_labels = [], []
class_counts = {c: 0 for c in range(10)}
for img, label in full_train_dataset:
    label = int(label)
    if class_counts[label] < 50:
        viz_images.append(img)
        viz_labels.append(label)
        class_counts[label] += 1
    if all(v >= 50 for v in class_counts.values()):
        break
viz_images = torch.stack(viz_images)
print(f"Viz samples: {len(viz_labels)} total, shape={viz_images.shape}")

Input samples: labels=[5, 0, 4, 1, 9], shape=torch.Size([5, 784])
Viz samples: 500 total, shape=torch.Size([500, 784])


## 4. Create Tracker

In [5]:
tracker = ExperimentTracker(
    run_id="mnist_centroid_all",
    model_name="MNIST +Centroid(All)",
    description="All standard + CentroidProjection on both hidden layers",
    hyperparameters={
        "lr": 0.001, "batch_size": 512, "epochs": 10,
        "weight_decay": 1e-4, "l1_lambda": 1e-4, "dropout": 0.3,
        "proj_alpha": PROJ_ALPHA, "proj_beta": PROJ_BETA,
    },
    model=model,
)

tracker.track("input", size=784)
tracker.track("hidden_1", model.fc1, size=128)
tracker.track("proj_1", model.proj1, size=128)
tracker.track("hidden_2", model.fc2, size=64)
tracker.track("proj_2", model.proj2, size=64)
tracker.track("output", model.fc3, size=10)

tracker.set_input_samples(five_images, five_labels)
tracker.set_viz_samples(viz_images, viz_labels)
tracker.enable_gradient_capture()
tracker.enable_forward_labels()
tracker.enable_loss_landscape()

ExperimentTracker: will write to /Users/morgancooper/NeuralNetworkProjectionOperator/scripts/../experimentation/runs/mnist_centroid_all_v1


## 5. Train

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
l1_lambda = 1e-4

CHECKPOINT_EVERY = 2
num_epochs = 10

def evaluate(loader):
    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            total_loss += criterion(out, y).item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

step = 0
global_batch = 0

for epoch in range(num_epochs):
    c1, c2 = model.compute_centroids(train_loader, device)
    model.proj1.update_centroids(c1)
    model.proj2.update_centroids(c2)
    print(f"Epoch {epoch}: centroids updated from full dataset")

    model.train()
    running_loss = 0.0

    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(batch_x, labels=batch_y)
        l1_norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(output, batch_y) + l1_lambda * l1_norm
        loss.backward()
        tracker.capture_gradients()
        optimizer.step()
        running_loss += loss.item()
        global_batch += 1
        if global_batch % CHECKPOINT_EVERY == 0:
            val_loss, val_acc = evaluate(val_loader)
            _, test_acc = evaluate(test_loader)
            tracker.compute_loss_landscape(batch_x, batch_y, criterion)
            tracker.save_checkpoint(step=step, epoch=epoch, metrics={
                "train_loss": running_loss / (batch_idx + 1),
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "test_accuracy": test_acc,
            })
            step += 1
            model.train()

    print(f"Epoch {epoch}: loss={running_loss / len(train_loader):.4f}")

print()
print(f"Total checkpoints saved: {step}")

Epoch 0: centroids updated from full dataset
  step_000.json (epoch=0, loss=1.9253, acc=0.4484, size=5.1MB)
  step_001.json (epoch=0, loss=1.6481, acc=0.6587, size=5.1MB)
  step_002.json (epoch=0, loss=1.4791, acc=0.7389, size=5.1MB)
  step_003.json (epoch=0, loss=1.3644, acc=0.7767, size=5.1MB)
  step_004.json (epoch=0, loss=1.2738, acc=0.8016, size=5.1MB)
  step_005.json (epoch=0, loss=1.1949, acc=0.8193, size=5.1MB)
  step_006.json (epoch=0, loss=1.1228, acc=0.8321, size=5.1MB)
  step_007.json (epoch=0, loss=1.0573, acc=0.8387, size=5.1MB)
  step_008.json (epoch=0, loss=0.9966, acc=0.8452, size=5.1MB)
  step_009.json (epoch=0, loss=0.9404, acc=0.8513, size=5.1MB)
  step_010.json (epoch=0, loss=0.8882, acc=0.8593, size=5.1MB)
  step_011.json (epoch=0, loss=0.8412, acc=0.8638, size=5.1MB)
  step_012.json (epoch=0, loss=0.7983, acc=0.8676, size=5.1MB)
  step_013.json (epoch=0, loss=0.7598, acc=0.8710, size=5.1MB)
  step_014.json (epoch=0, loss=0.7242, acc=0.8750, size=5.1MB)
  step_015

## 6. Finalize

In [7]:
tracker.finalize()
print(f"Run directory: {tracker.run_dir}")
print(f"Run ID: {tracker.run_id}")

Finalized run 'mnist_centroid_all_v1' with 470 checkpoints
Run directory: /Users/morgancooper/NeuralNetworkProjectionOperator/scripts/../experimentation/runs/mnist_centroid_all_v1
Run ID: mnist_centroid_all_v1
